In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
train_df = pd.read_parquet("C:/Users/mojes/Desktop/IDS PROJECT/dataset/UNSW_NB15_training-set.parquet")
test_df  = pd.read_parquet("C:/Users/mojes/Desktop/IDS PROJECT/dataset/UNSW_NB15_testing-set.parquet")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (175341, 36)
Test shape: (82332, 36)


In [3]:
train_df.head()
train_df.columns

Index(['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes',
       'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt',
       'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt',
       'synack', 'ackdat', 'smean', 'dmean', 'trans_depth',
       'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports',
       'attack_cat', 'label'],
      dtype='str')

In [16]:
y_train = train_df['label']
y_test  = test_df['label']

X_train = train_df.drop(columns=['label', 'attack_cat'])
X_test  = test_df.drop(columns=['label', 'attack_cat'])

In [17]:
from sklearn.preprocessing import LabelEncoder

label_encoders_unsw = {}

categorical_cols = ['proto', 'service', 'state']

for col in categorical_cols:
    le = LabelEncoder()
    
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    
    # Handle unseen labels safely
    X_test[col] = X_test[col].astype(str).map(
        lambda s: le.transform([s])[0] if s in le.classes_ else -1
    )
    
    label_encoders_unsw[col] = le

In [18]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Nulls:", X_train.isnull().sum().sum())

Train shape: (175341, 34)
Test shape: (82332, 34)
Nulls: 0


In [23]:
import lightgbm as lgb

unsw_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

unsw_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 119341, number of negative: 56000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012251 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5853
[LightGBM] [Info] Number of data points in the train set: 175341, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680622 -> initscore=0.756633
[LightGBM] [Info] Start training from score 0.756633


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [24]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = unsw_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("UNSW Accuracy:", acc)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

UNSW Accuracy: 0.8641111596948938

Confusion Matrix:
 [[26799 10201]
 [  987 44345]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.72      0.83     37000
           1       0.81      0.98      0.89     45332

    accuracy                           0.86     82332
   macro avg       0.89      0.85      0.86     82332
weighted avg       0.88      0.86      0.86     82332



In [9]:
TP = cm[1,1]
TN = cm[0,0]
FP = cm[0,1]
FN = cm[1,0]

FPR = FP / (FP + TN)
FNR = FN / (FN + TP)
Detection_Rate = TP / (TP + FN)

print(f"FPR: {FPR:.6f}")
print(f"FNR: {FNR:.6f}")
print(f"Detection Rate: {Detection_Rate:.6f}")

FPR: 0.275703
FNR: 0.021773
Detection Rate: 0.978227


In [10]:
y_train_multi = train_df['attack_cat']
y_test_multi  = test_df['attack_cat']

In [11]:
from sklearn.preprocessing import LabelEncoder

le_attack = LabelEncoder()

y_train_multi = le_attack.fit_transform(y_train_multi.astype(str))
y_test_multi  = le_attack.transform(y_test_multi.astype(str))

In [12]:
unsw_multi_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    objective='multiclass',
    num_class=len(np.unique(y_train_multi)),
    random_state=42
)

unsw_multi_model.fit(X_train, y_train_multi)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012899 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5853
[LightGBM] [Info] Number of data points in the train set: 175341, number of used features: 34
[LightGBM] [Info] Start training from score -4.473585
[LightGBM] [Info] Start training from score -4.609405
[LightGBM] [Info] Start training from score -2.660065
[LightGBM] [Info] Start training from score -1.658386
[LightGBM] [Info] Start training from score -2.266191
[LightGBM] [Info] Start training from score -1.477853
[LightGBM] [Info] Start training from score -1.141381
[LightGBM] [Info] Start training from score -2.816215
[LightGBM] [Info] Start training from score -5.041864
[LightGBM] [Info] Start training from score -7.206953


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [13]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

y_pred_multi = unsw_multi_model.predict(X_test)

print("Multiclass Accuracy:", accuracy_score(y_test_multi, y_pred_multi))
print("\nClassification Report:\n", classification_report(y_test_multi, y_pred_multi))

cm_multi = confusion_matrix(y_test_multi, y_pred_multi)
print("\nConfusion Matrix:\n", cm_multi)

Multiclass Accuracy: 0.7844580479036097

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       677
           1       0.12      0.06      0.08       583
           2       0.59      0.08      0.13      4089
           3       0.57      0.90      0.70     11132
           4       0.34      0.56      0.42      6062
           5       1.00      0.97      0.98     18871
           6       0.94      0.80      0.86     37000
           7       0.91      0.80      0.85      3496
           8       0.41      0.63      0.50       378
           9       0.04      0.32      0.07        44

    accuracy                           0.78     82332
   macro avg       0.49      0.51      0.46     82332
weighted avg       0.83      0.78      0.79     82332


Confusion Matrix:
 [[    2     5     6   651     1     0    12     0     0     0]
 [    2    36     6   524     1     6     4     1     3     0]
 [    3    64   311  3446    9

In [25]:
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

param_grid = {
    'num_leaves': [31, 50],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'max_depth': [-1, 10]
}

model = lgb.LGBMClassifier(random_state=42)

grid = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring='f1',
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Number of positive: 119341, number of negative: 56000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012844 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5853
[LightGBM] [Info] Number of data points in the train set: 175341, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680622 -> initscore=0.756633
[LightGBM] [Info] Start training from score 0.756633
Best Params: {'learning_rate': 0.1, 'max_depth': -1, 'n_estimators': 200, 'num_leaves': 31}


In [26]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'num_leaves': [20, 31, 50, 70],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'max_depth': [-1, 10, 20],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

random_search = RandomizedSearchCV(
    lgb.LGBMClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='f1',
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Params:", random_search.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[LightGBM] [Info] Number of positive: 119341, number of negative: 56000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012869 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5853
[LightGBM] [Info] Number of data points in the train set: 175341, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680622 -> initscore=0.756633
[LightGBM] [Info] Start training from score 0.756633
Best Params: {'subsample': 1.0, 'num_leaves': 20, 'n_estimators': 300, 'max_depth': 20, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [27]:
unsw_tuned_model = lgb.LGBMClassifier(
    subsample=1.0,
    num_leaves=20,
    n_estimators=300,
    max_depth=20,
    learning_rate=0.1,
    colsample_bytree=1.0,
    random_state=42
)

unsw_tuned_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 119341, number of negative: 56000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014255 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5853
[LightGBM] [Info] Number of data points in the train set: 175341, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680622 -> initscore=0.756633
[LightGBM] [Info] Start training from score 0.756633


,boosting_type,'gbdt'
,num_leaves,20
,max_depth,20
,learning_rate,0.1
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [28]:
y_pred_tuned = unsw_tuned_model.predict(X_test)

acc = accuracy_score(y_test, y_pred_tuned)
cm = confusion_matrix(y_test, y_pred_tuned)

print("Tuned Accuracy:", acc)
print("Confusion Matrix:\n", cm)

Tuned Accuracy: 0.8711922460282757
Confusion Matrix:
 [[27525  9475]
 [ 1130 44202]]


In [29]:
TP = cm[1,1]
TN = cm[0,0]
FP = cm[0,1]
FN = cm[1,0]

FPR = FP / (FP + TN)
FNR = FN / (FN + TP)
Detection_Rate = TP / (TP + FN)

print(f"FPR: {FPR:.6f}")
print(f"FNR: {FNR:.6f}")
print(f"Detection Rate: {Detection_Rate:.6f}")

FPR: 0.256081
FNR: 0.024927
Detection Rate: 0.975073


In [30]:
unsw_model = lgb.LGBMClassifier(
    class_weight='balanced',
    n_estimators=300,
    learning_rate=0.1,
    num_leaves=20,
    max_depth=20
)

In [31]:
importances = unsw_model.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns)

top_features = feat_imp.sort_values(ascending=False).head(20).index

X_train_fs = X_train[top_features]
X_test_fs = X_test[top_features]

NotFittedError: No feature_importances found. Need to call fit beforehand.

In [32]:
import pandas as pd

importances = unsw_tuned_model.feature_importances_

feat_imp = pd.Series(importances, index=X_train.columns)

# Sort features
feat_imp_sorted = feat_imp.sort_values(ascending=False)

print(feat_imp_sorted.head(20))

sbytes              707
smean               561
ackdat              298
synack              288
tcprtt              255
ct_src_dport_ltm    245
dbytes              242
dtcpb               234
sload               227
dload               226
stcpb               217
dur                 207
sinpkt              206
proto               205
dmean               186
ct_dst_sport_ltm    159
rate                150
djit                141
sjit                138
service             109
dtype: int32


In [33]:
top_features = feat_imp_sorted.head(20).index

In [34]:
X_train_fs = X_train[top_features]
X_test_fs  = X_test[top_features]

In [35]:
import lightgbm as lgb

unsw_final_model = lgb.LGBMClassifier(
    subsample=1.0,
    num_leaves=20,
    n_estimators=300,
    max_depth=20,
    learning_rate=0.1,
    colsample_bytree=1.0,
    random_state=42
)

unsw_final_model.fit(X_train_fs, y_train)

[LightGBM] [Info] Number of positive: 119341, number of negative: 56000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004895 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4305
[LightGBM] [Info] Number of data points in the train set: 175341, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680622 -> initscore=0.756633
[LightGBM] [Info] Start training from score 0.756633


,boosting_type,'gbdt'
,num_leaves,20
,max_depth,20
,learning_rate,0.1
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [36]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred_final = unsw_final_model.predict(X_test_fs)

acc_final = accuracy_score(y_test, y_pred_final)
cm_final = confusion_matrix(y_test, y_pred_final)

print("Final Accuracy:", acc_final)
print("Confusion Matrix:\n", cm_final)
print("\nClassification Report:\n", classification_report(y_test, y_pred_final))

Final Accuracy: 0.8673784190837098
Confusion Matrix:
 [[27192  9808]
 [ 1111 44221]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.73      0.83     37000
           1       0.82      0.98      0.89     45332

    accuracy                           0.87     82332
   macro avg       0.89      0.86      0.86     82332
weighted avg       0.88      0.87      0.86     82332



In [37]:
from sklearn.preprocessing import LabelEncoder

le_attack = LabelEncoder()

y_train_multi = le_attack.fit_transform(train_df['attack_cat'].astype(str))
y_test_multi  = le_attack.transform(test_df['attack_cat'].astype(str))

In [39]:
from sklearn.model_selection import RandomizedSearchCV
import lightgbm as lgb

param_dist = {
    'num_leaves': [20, 31, 50],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 300],
    'max_depth': [-1, 10, 20],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

multi_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(set(y_train_multi)),
    random_state=42
)

random_search_multi = RandomizedSearchCV(
    multi_model,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',   # IMPORTANT
    verbose=1,
    n_jobs=-1
)

random_search_multi.fit(X_train, y_train_multi)

print("Best Params:", random_search_multi.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015800 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5853
[LightGBM] [Info] Number of data points in the train set: 175341, number of used features: 34
[LightGBM] [Info] Start training from score -4.473585
[LightGBM] [Info] Start training from score -4.609405
[LightGBM] [Info] Start training from score -2.660065
[LightGBM] [Info] Start training from score -1.658386
[LightGBM] [Info] Start training from score -2.266191
[LightGBM] [Info] Start training from score -1.477853
[LightGBM] [Info] Start training from score -1.141381
[LightGBM] [Info] Start training from score -2.816215
[LightGBM] [Info] Start training from score -5.041864
[LightGBM] [Info] Start training from score -7.206953
Best Params: {'subsample': 0.8, 'num_leaves':

In [40]:
unsw_multi_final = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(set(y_train_multi)),
    subsample=0.8,
    num_leaves=50,
    n_estimators=300,
    max_depth=20,
    learning_rate=0.05,
    colsample_bytree=1.0,
    random_state=42
)

unsw_multi_final.fit(X_train, y_train_multi)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007535 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5853
[LightGBM] [Info] Number of data points in the train set: 175341, number of used features: 34
[LightGBM] [Info] Start training from score -4.473585
[LightGBM] [Info] Start training from score -4.609405
[LightGBM] [Info] Start training from score -2.660065
[LightGBM] [Info] Start training from score -1.658386
[LightGBM] [Info] Start training from score -2.266191
[LightGBM] [Info] Start training from score -1.477853
[LightGBM] [Info] Start training from score -1.141381
[LightGBM] [Info] Start training from score -2.816215
[LightGBM] [Info] Start training from score -5.041864
[LightGBM] [Info] Start training from score -7.206953


,boosting_type,'gbdt'
,num_leaves,50
,max_depth,20
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [41]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_multi = unsw_multi_final.predict(X_test)

acc_multi = accuracy_score(y_test_multi, y_pred_multi)

print("Tuned Multiclass Accuracy:", acc_multi)
print("\nClassification Report:\n", classification_report(y_test_multi, y_pred_multi))

Tuned Multiclass Accuracy: 0.7508380702521499

Classification Report:
               precision    recall  f1-score   support

           0       0.04      0.05      0.04       677
           1       0.06      0.04      0.05       583
           2       0.31      0.11      0.16      4089
           3       0.58      0.81      0.67     11132
           4       0.29      0.53      0.38      6062
           5       0.98      0.96      0.97     18871
           6       0.93      0.76      0.84     37000
           7       0.81      0.74      0.77      3496
           8       0.30      0.58      0.39       378
           9       0.06      0.45      0.11        44

    accuracy                           0.75     82332
   macro avg       0.44      0.50      0.44     82332
weighted avg       0.79      0.75      0.76     82332



In [42]:
canonical_map = {
    'duration': ['duration', 'dur'],
    'src_bytes': ['src_bytes', 'sbytes'],
    'dst_bytes': ['dst_bytes', 'dbytes'],
    'src_pkts': ['src_pkts', 'spkts'],
    'dst_pkts': ['dst_pkts', 'dpkts'],
    'protocol': ['proto'],
    'service': ['service']
}